In [0]:

# Notebook : helper_functions
# Purpose  : Reusable helper functions for Procurement Project
# Author   : V R  Mutyala


from pyspark.sql import DataFrame
from pyspark.sql.functions import *
from pyspark.sql.window import Window


# Preview Data
def preview(df: DataFrame, table_name: str, rows: int = 10):
    """
    Display row count and sample records.
    """

    print("=" * 60)
    print(f"Table : {table_name}")
    print(f"Rows  : {df.count()}")
    print("=" * 60)

    display(df.limit(rows))

#Record Count
def record_count(df: DataFrame, table_name: str):
    """
    Print record count.
    """

    print(f"{table_name} : {df.count():,} records")

# Null Check 
def null_check(df: DataFrame):
    """
    Count null values for every column.
    """

    return df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df.columns
    ])

# Duplicate Check 
def duplicate_check(df: DataFrame, key_columns: list):
    """
    Returns duplicate records.
    """

    duplicates = (
        df.groupBy(key_columns)
          .count()
          .filter(col("count") > 1)
    )

    return duplicates

#Sepreate Duplicates
def split_duplicates(df: DataFrame, key_columns: list):

    duplicate_keys = (
        df.groupBy(key_columns)
          .count()
          .filter(col("count") > 1)
          .drop("count")
    )

    duplicates = df.join(
        duplicate_keys,
        key_columns,
        "inner"
    )

    valid_records = df.join(
        duplicate_keys,
        key_columns,
        "left_anti"
    )

    return valid_records, duplicates

#Write Delta table
def write_delta(
    df: DataFrame,
    table_name: str,
    mode: str = "overwrite"
):
    """
    Write dataframe to Delta table.
    """

    (
        df.write
          .format("delta")
          .mode(mode)
          .saveAsTable(table_name)
    )

    print(f"Table written successfully : {table_name}")

# Read Delta table
def read_delta(table_name: str):
    """
    Read Delta table.
    """

    return spark.table(table_name)

# Add Audit columns 
def add_audit_columns(df: DataFrame):

    return (
        df
        .withColumn("load_timestamp", current_timestamp())
        .withColumn("created_by", lit("Databricks"))
    )

#Data Quality Summary 
def data_quality_summary(df: DataFrame):

    print("=" * 60)
    print(f"Rows       : {df.count():,}")
    print(f"Columns    : {len(df.columns)}")
    print("=" * 60)

    null_check(df).show() 
    
